[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/02-python-for-data-science/pyds-manipulation.ipynb)

# Data Manipulation with Pandas

*AIBits Academy · Machine Learning End To End · Python For Data Science*

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

The four everyday moves on a clean table: **sorting** to order it, **filtering** to subset it, and **grouping + aggregation** to summarise it. Together they answer most "what does the data say?" questions.

Every example on this page uses one small sales table — eight rows across three regions and two departments.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'Region':     ['East', 'West', 'East', 'North', 'West', 'North', 'East', 'West'],
    'Department': ['Apparel', 'Grocery', 'Apparel', 'Grocery', 'Apparel', 'Grocery', 'Grocery', 'Apparel'],
    'Sales':      [620, 340, 780, 210, 560, 430, 690, 300],
    'Price':      [1200, 300, 1500, 250, 1100, 320, 280, 900],
    'Quantity':   [5, 8, 6, 9, 4, 7, 10, 3],
})

## Sorting

`sort_values` orders rows by one or more columns. Pass a list to sort by several keys, and a matching list of booleans to mix ascending and descending.

In [ ]:
print("by Sales, highest first:\n", df.sort_values(by='Sales', ascending=False).head())

print("\nby Region (A-Z), then Sales (high-low):\n",
      df.sort_values(by=['Region', 'Sales'], ascending=[True, False]).head())

## Filtering

A boolean condition inside `df[...]` keeps only the matching rows. Combine conditions with `&` (and) / `|` (or) — each condition in parentheses — or use `.isin()` to match a list of values.

In [ ]:
print("Quantity > 5:\n", df[df['Quantity'] > 5])

print("\nEast region AND Sales > 650:\n",
      df[(df['Region'] == 'East') & (df['Sales'] > 650)])

print("\nRegion in East or North:\n",
      df[df['Region'].isin(['East', 'North'])])

## Grouping & Aggregation

The **split-apply-combine** pattern is the heart of pandas analysis: `groupby` *splits* rows into groups, an aggregation is *applied* to each group, and the results are *combined* into one summary table.

Pass a dictionary-style aggregation to compute several summaries at once, giving each a clear output name.

In [ ]:
grouped = df.groupby('Region').agg(
    Total_Sales=('Sales', 'sum'),
    Avg_Price=('Price', 'mean'),
    Total_Qty=('Quantity', 'sum'),
)
print(grouped.round(1))

Group by several columns for a finer breakdown, or aggregate a categorical column with its `mode` (most frequent value).

In [ ]:
print("sales by Region and Department:\n",
      df.groupby(['Region', 'Department'])['Sales'].sum())

def most_common(s):
    return s.mode().iloc[0]
print("\nmost common department per region:\n",
      df.groupby('Region')['Department'].agg(most_common))

### transform — Group Stats Back on Every Row

Where `agg` collapses each group to one row, `transform` returns a value for *every* original row — perfect for adding a "share of group total" column.

In [ ]:
df['RegionTotal'] = df.groupby('Region')['Sales'].transform('sum')
df['PctOfRegion'] = (df['Sales'] / df['RegionTotal'] * 100).round(1)
print(df[['Region', 'Sales', 'RegionTotal', 'PctOfRegion']])

### Two Quick Questions

The top three sales rows, and the average sale per department.

In [ ]:
print("top 3 by Sales:\n", df.nlargest(3, 'Sales')[['Region', 'Department', 'Sales']])
print("\navg sales per department:\n",
      df.groupby('Department')['Sales'].mean().round(1))

> **✅ What You Can Now Do**
>
> You can sort by multiple keys, filter with compound conditions and `.isin()`, and run the split-apply-combine pattern with `groupby` + `agg`/`transform` to summarise data any way you need. Next: advanced label- and position-based selection with `.loc` and `.iloc`.

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Top three with a filter

Using the store table `df` from the lesson, find the 3 highest-`Sales` rows **in the West region**. Store the result in `top_west`.

In [ ]:
top_west = None   # TODO (df comes from the first code cell of the lesson)


In [ ]:
try:
    check("three rows", top_west is not None and len(top_west) == 3)
    check("all West", set(top_west["Region"]) == {"West"})
    check("sorted by sales", top_west["Sales"].tolist() == [560, 340, 300])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
top_west = df[df["Region"] == "West"].nlargest(3, "Sales")

```

</details>

### Exercise 2 · Medium · Aggregate with named columns

Build `dept_summary`, one row per `Department`, with columns `Revenue` (sum of `Sales`), `AvgQty` (mean `Quantity`) and `Orders` (row count), using `groupby(...).agg(...)` with named aggregations.

In [ ]:
dept_summary = None   # TODO


In [ ]:
try:
    check("two departments", dept_summary is not None and len(dept_summary) == 2)
    check("columns", list(dept_summary.columns) == ["Revenue", "AvgQty", "Orders"])
    check("Apparel revenue", dept_summary.loc["Apparel", "Revenue"] == 2260)
    check("Grocery orders", dept_summary.loc["Grocery", "Orders"] == 4)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
dept_summary = df.groupby("Department").agg(
    Revenue=("Sales", "sum"),
    AvgQty=("Quantity", "mean"),
    Orders=("Sales", "size"),
)

```

</details>

### Exercise 3 · Stretch · Rank within a group

Add a column `RankInRegion` to a copy `ranked` of `df`: 1 for the highest `Sales` row in each region, 2 for the next, and so on (`groupby(...).rank(ascending=False, method="first")`, as an integer).

In [ ]:
ranked = None   # TODO


In [ ]:
try:
    check("column exists", ranked is not None and "RankInRegion" in ranked.columns)
    check("best East row ranks 1", ranked[(ranked["Region"] == "East") & (ranked["Sales"] == 780)]["RankInRegion"].iloc[0] == 1)
    check("each region has a rank 1", ranked.groupby("Region")["RankInRegion"].min().eq(1).all())
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
ranked = df.copy()
ranked["RankInRegion"] = ranked.groupby("Region")["Sales"].rank(ascending=False, method="first").astype(int)

```

</details>

---
*Back to the course: **Machine Learning End To End → Data Manipulation with Pandas**.*